# iPro-MP External Benchmark

This notebook prepares and evaluates the iPro-MP external baseline for the shared SeqTrainer promoter benchmark.

It uses the same split and metric policy as CNN-v2 and DNABERT2:

- same train/validation/test CSV files
- seed `42`
- validation-only threshold selection using MCC
- test set used only for final reporting
- primary metric: MCC
- secondary metric: AUPRC

Important: this notebook does not fake iPro-MP metrics. If official iPro-MP prediction files are missing, it writes FASTA/mapping files and records a skipped benchmark manifest.


## Where Should This Run?

Use this notebook on **Colab or local** for SeqTrainer-side preparation and evaluation. These steps only need Python, pandas, and the SeqTrainer package.

Use a **local Linux/HPC conda environment** for the official iPro-MP prediction command. That is the better route because official iPro-MP expects Python 3.8, DNABERT-6, downloaded Zenodo model weights, and local paths such as `DNABERT-6`, `models`, and `Predict_Results`.

Colab can prepare FASTA files, but it is less ideal for the official iPro-MP run because the runtime is temporary and may not match the official Python/dependency stack.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

import pandas as pd

IS_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IS_COLAB)


## Repository Setup

In Colab, this cell clones the current benchmark branch. Locally or in VS Code, it finds the repository root from the notebook location.


In [ ]:
REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"

if IS_COLAB:
    REPO_DIR = Path("/content/SeqTrainer")
    if not REPO_DIR.exists():
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    REPO_DIR = next((path for path in candidates if (path / "pyproject.toml").exists() and (path / "src" / "seqtrainer").exists()), current)
    os.chdir(REPO_DIR)

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
try:
    print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
    print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
except Exception as exc:
    print("Git metadata unavailable:", exc)


## Install SeqTrainer

Run this in Colab or a fresh local environment. If you already installed the package locally, this is harmless but may take a minute.


In [ ]:
def import_seqtrainer_tools():
    from seqtrainer.benchmarks import load_benchmark_config
    from seqtrainer.adapters.ipromp import prepare_ipromp_inputs
    from seqtrainer.benchmarks.runner import run_benchmark
    return load_benchmark_config, prepare_ipromp_inputs, run_benchmark

try:
    load_benchmark_config, prepare_ipromp_inputs, run_benchmark = import_seqtrainer_tools()
except ModuleNotFoundError as exc:
    print("Missing dependency while importing SeqTrainer:", exc)
    print("Installing SeqTrainer with benchmark dependencies, then retrying...")
    %pip install -q -e ".[torch]"
    load_benchmark_config, prepare_ipromp_inputs, run_benchmark = import_seqtrainer_tools()

print("SeqTrainer imports OK")


## Prepare Shared Split Data

The iPro-MP benchmark must use the exact same CSV split files as CNN-v2 and DNABERT2.

This cell uses the bundled repo ZIP if the CSVs are not already extracted.


In [ ]:
DATA_DIR = REPO_DIR / "data" / "promoter_classification"
ZIP_PATH = REPO_DIR / "data" / "data_DNABERT" / "promoter_classification_DNABERT.zip"
SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
missing = [name for name in SPLIT_FILES.values() if not (DATA_DIR / name).exists()]
if missing:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Missing split CSVs and bundled ZIP was not found: {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = set(zf.namelist())
        for file_name in SPLIT_FILES.values():
            if file_name not in names:
                raise FileNotFoundError(f"{file_name} not found inside {ZIP_PATH}")
            with zf.open(file_name) as src, (DATA_DIR / file_name).open("wb") as dst:
                shutil.copyfileobj(src, dst)
            print("Extracted", file_name)
else:
    print("Shared split CSVs already exist")

for split, file_name in SPLIT_FILES.items():
    frame = pd.read_csv(DATA_DIR / file_name)
    print(split, frame.shape, frame["label"].value_counts().sort_index().to_dict())


## Load iPro-MP Config

The config records the iPro-MP species, official repo, Zenodo model source, DNABERT-6 path, FASTA output paths, prediction paths, and shared metric policy.


In [ ]:
CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "ipromp_external.toml"
config = load_benchmark_config(CONFIG)
print("Experiment:", config.experiment.name)
print("Model:", config.model.name)
print("Species:", config.model.params.get("species_name"), "species_id=", config.model.params.get("species_id"))
print("Official repo:", config.model.params.get("official_repo"))
print("Pretrained model source:", config.model.params.get("pretrained_model_source"))
print("Threshold strategy:", config.evaluation.threshold_strategy)


## Prepare FASTA, Mapping, And Official Run Commands

This step does not require iPro-MP, DNABERT-6, or pretrained weights. It only converts SeqTrainer's shared CSV splits into iPro-MP-compatible FASTA files and a mapping table.


In [ ]:
prepared = prepare_ipromp_inputs(CONFIG, base_dir=REPO_DIR)
print("output_dir:", prepared.output_dir)
print("mapping_csv:", prepared.mapping_csv)
print("command_script:", prepared.command_script)
print("external_prediction_schema:", prepared.prediction_schema)
for split, path in prepared.fasta_paths.items():
    print(f"{split}: {path}")


## Inspect FASTA And Mapping

The FASTA header preserves stable row identity. Official iPro-MP outputs only `Sequence`, so the mapping CSV is required to recover split, label, and row order.


In [ ]:
with prepared.fasta_paths["validation"].open("r", encoding="utf-8") as handle:
    preview = [next(handle).rstrip() for _ in range(4)]
print("Validation FASTA preview:")
print("\n".join(preview))

mapping = pd.read_csv(prepared.mapping_csv)
print("Mapping shape:", mapping.shape)
display(mapping.head())
print(mapping.groupby("split")["label"].value_counts().unstack(fill_value=0))


## Official iPro-MP Commands

Run these commands in the official iPro-MP environment after downloading DNABERT-6 and the Zenodo pretrained models. For E. coli K-12 MG1655, `species_id = 10`.


In [ ]:
print(prepared.command_script.read_text(encoding="utf-8"))


## Run SeqTrainer iPro-MP Benchmark

If official prediction CSVs are missing, this writes a skipped manifest and does not create fake metrics.

If the configured validation/test prediction CSVs exist, this cell normalizes official iPro-MP predictions, selects the threshold on validation MCC, applies it to test, and writes shared artifacts.


In [ ]:
result = run_benchmark(CONFIG, base_dir=REPO_DIR, allow_skip=True)
print("status:", result.status)
print("output_dir:", result.output_dir)
print("manifest extra:")
print(json.dumps(result.manifest.get("extra", {}), indent=2))


## Metrics Or Next Action

If metrics exist, display them. Otherwise, use the generated FASTA files with the official iPro-MP prediction script, copy the resulting CSVs into `external_predictions/`, then rerun the previous cell.


In [ ]:
OUTPUT_DIR = REPO_DIR / "outputs" / "benchmarks" / "ipromp_external_ep_genomic_order"
metrics_path = OUTPUT_DIR / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
else:
    print("No iPro-MP metrics yet because official prediction files are missing.")
    print("This is expected for a preparation-only run.")
    print("Expected prediction paths:")
    print(config.model.params.get("validation_predictions_csv"))
    print(config.model.params.get("test_predictions_csv"))


## Compare After Predictions Exist

After iPro-MP metrics are available, compare it against CNN-v2 and DNABERT2 with:

```bash
seqtrainer benchmark compare \
  outputs/benchmarks/cnn_v2_regularized_ep_genomic_order \
  outputs/benchmarks/dnabert2_frozen_ep_genomic_order \
  outputs/benchmarks/ipromp_external_ep_genomic_order \
  --output-dir outputs/benchmarks/comparison
```
